In [6]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# Descargar cierres SAN.MC último año
end = datetime.now()
start = end - timedelta(days=1825)  # Aproximadamente 5 años
ams = yf.download('TSLA', start=start, end=end)['Close'].reset_index()
ams.columns = ['Date', 'Close_TSLA']
ams['Date'] = pd.to_datetime(ams['Date']).dt.date
ams.to_csv('TSLA_cierres.csv', index=False)
print("✅ TSLA_cierres.csv guardado!")
print(ams.tail())

[*********************100%***********************]  1 of 1 completed

✅ TSLA_cierres.csv guardado!
            Date  Close_TSLA
1250  2026-02-12  417.070007
1251  2026-02-13  417.440002
1252  2026-02-17  410.630005
1253  2026-02-18  411.320007
1254  2026-02-19  411.709991


In [26]:

    # 1. DATOS SAN.MC (1 año)
    end = datetime.now()
    start = end - timedelta(days=1825)  # Aproximadamente 5 años
    df = yf.download('TSLA', start=start, end=end)['Close'].reset_index()
    df.columns = ['Date', 'Close']
    df.to_csv('TSLA_data.csv', index=False)


    # 2. BACKTEST CORREGIDO
    df['Return'] = df['Close'].pct_change() * 100
    capital = 10000
    position = 0  # 0 = cash, 1 = comprado
    buy_price = 0
    buy_day = 0
    operaciones = []


    for i in range(1, len(df)):
        today_price = df['Close'].iloc[i]
        today_ret = df['Return'].iloc[i]
    
        # SEÑAL COMPRA: -5% y sin posición
        if today_ret <= -5 and position == 0:
            position = 1
            buy_price = df['Close'].iloc[i]  # Precio cierre día caída
            buy_day = i
            capital_invertido = capital
            capital = 0
            operaciones.append(['BUY', df['Date'].iloc[i], buy_price, capital_invertido])
    
        # SI COMPRADO: check día SIGUIENTE (i = buy_day + 1)
        elif position == 1 and i == buy_day + 1:
            next_day_ret = (today_price / buy_price - 1) * 100
        
            if next_day_ret >= 3:  # +2% → VENTA GANADORA
                capital = capital_invertido * (today_price / buy_price)
                operaciones.append(['SELL +2%', df['Date'].iloc[i], today_price, capital])
                position = 0
            else:  # NO +2% → VENTA al cierre día siguiente
                today_price = df['Close'].iloc[i+7]  # Precio cierre día siguiente
                capital = capital_invertido * (today_price / buy_price)
                operaciones.append(['SELL cierre', df['Date'].iloc[i+7], today_price, capital])
                position = 0


    # Posición final
    if position == 1:
        capital = capital_invertido * (df['Close'].iloc[-1] / buy_price)


    print(f"💰 INICIAL: €10.000")
    print(f"💵 FINAL: €{capital:,.0f}")
    print(f"📈 RENDIMIENTO: {(capital/10000-1)*100:+.1f}%")
    print(f"🔢 OPERACIONES: {len(operaciones)}")


    pd.DataFrame(operaciones, columns=['Tipo', 'Fecha', 'Precio', 'Capital']).to_csv('TSLA_backtest_corregido.csv', index=False)
    df.to_csv('TSLA_completo.csv', index=False)


    print("\n✅ Archivos:")

[*********************100%***********************]  1 of 1 completed

💰 INICIAL: €10.000
💵 FINAL: €9,456
📈 RENDIMIENTO: -5.4%
🔢 OPERACIONES: 180

✅ Archivos:


In [48]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np  # Para NaNs
import yfinance as yf
# 1. DATOS TSLA (5 años) - Carga o descarga
end = datetime.now()
start = end - timedelta(days=1825)  # Aproximadamente 5 años
df = yf.download('ASML', start=start, end=end)['Close'].reset_index()
df.columns = ['Date', 'Close']
df.to_csv('ASML_data.csv', index=False)
    
    

df = df.sort_values('Date').reset_index(drop=True)
df['Return'] = df['Close'].pct_change() * 100

# 2. BACKTEST: Escalera rentabilidad decreciente D1-D7
capital = 10000.0
position = 0  # 0 = cash, 1 = comprado
buy_price = 0.0
buy_day = -1
operaciones = []

# % requeridos por día: [D1:5%, D2:4%, D3:3%, D4:3%, D5:2%, D6:1%, D7:cualquier]
sell_thresholds = [2, 2, 2, 2, 2, 1, 0]  # Día 7: 0% = vende siempre

for i in range(1, len(df)):
    today_price = df['Close'].iloc[i]
    today_ret = df['Return'].iloc[i]
   
    # SEÑAL COMPRA: -3% y sin posición
    if today_ret <= -3 and position == 0:
        position = 1
        buy_price = df['Close'].iloc[i]  # Precio cierre día caída
        buy_day = i
        capital_invertido = capital
        capital = 0
        operaciones.append(['BUY', df['Date'].iloc[i].date(), buy_price, capital_invertido])
    
    # Si comprado: check días 1-7 con % decreciente
    elif position == 1 and buy_day < i <= buy_day + 7:
        days_held = i - buy_day  # 1,2,3,...,7
        req_pct = sell_thresholds[days_held - 1]  # Índice 0-6
        ret_since_buy = (today_price / buy_price - 1) * 100
        
        if ret_since_buy >= req_pct:
            capital = capital_invertido * (today_price / buy_price)
            operaciones.append([f'SELL +{req_pct}% D{days_held}', df['Date'].iloc[i].date(), today_price, capital])
            position = 0
            buy_day = -1
        elif days_held == 7:  # Día 7: VENTA SIEMPRE
            capital = capital_invertido * (today_price / buy_price)
            operaciones.append(['SELL día 7', df['Date'].iloc[i].date(), today_price, capital])
            position = 0
            buy_day = -1

# Posición final (rara)
if position == 1:
    final_price = df['Close'].iloc[-1]
    capital = capital_invertido * (final_price / buy_price)
    operaciones.append(['SELL final', df['Date'].iloc[-1].date(), final_price, capital])

# RESULTADOS
print(f"💰 INICIAL: €10.000")
print(f"💵 FINAL: €{capital:,.0f}")
print(f"📈 RENDIMIENTO: {(capital/10000-1)*100:+.1f}%")
print(f"🔢 OPERACIONES: {len(operaciones)}")
print("\nEscalera aplicada:")
for d, pct in enumerate(sell_thresholds, 1):
    print(f"  Día {d}: +{pct}% mínimo")

# GUARDAR
pd.DataFrame(operaciones, columns=['Tipo', 'Fecha', 'Precio', 'Capital']).to_csv('ASML_backtest_escalera.csv', index=False)
df.to_csv('ASML_completo.csv', index=False)

print("\n✅ Archivos:")
print("- ASML_backtest_escalera.csv")
print("- ASML_completo.csv")
print("\nOperaciones:")
print(pd.DataFrame(operaciones, columns=['Tipo', 'Fecha', 'Precio', 'Capital']).tail(10))


[*********************100%***********************]  1 of 1 completed

💰 INICIAL: €10.000
💵 FINAL: €9,356
📈 RENDIMIENTO: -6.4%
🔢 OPERACIONES: 154

Escalera aplicada:
  Día 1: +2% mínimo
  Día 2: +2% mínimo
  Día 3: +2% mínimo
  Día 4: +2% mínimo
  Día 5: +2% mínimo
  Día 6: +1% mínimo
  Día 7: +0% mínimo

✅ Archivos:
- ASML_backtest_escalera.csv
- ASML_completo.csv

Operaciones:
            Tipo       Fecha       Precio      Capital
144          BUY  2025-10-07   999.206909  9029.921317
145  SELL +0% D7  2025-10-16  1016.443604  9185.690850
146          BUY  2025-11-04  1028.783081  9185.690850
147   SELL día 7  2025-11-13  1018.516541  9094.024036
148          BUY  2025-11-20   979.747681  9094.024036
149  SELL +2% D3  2025-11-25  1001.898438  9299.627498
150          BUY  2025-12-12  1079.426270  9299.627498
151   SELL día 7  2025-12-23  1060.441284  9136.065338
152          BUY  2026-02-03  1394.041260  9136.065338
153  SELL +2% D4  2026-02-09  1427.606934  9356.043181
